# Assignment 5: Advanced Autograd — Higher-Order Derivatives (100 points)

This assignment deepens your understanding of PyTorch's automatic differentiation engine. You will compute higher-order derivatives, Jacobians, Hessians, and Jacobian-vector products — all skills that appear in PINNs, second-order optimization, and advanced architectures.

## Background

PyTorch's `torch.autograd.grad` can compute arbitrary-order derivatives by chaining calls with `create_graph=True`. This enables:

- **PINNs:** Computing $u_{xx}$, $u_{tt}$, and higher derivatives for PDE residuals
- **Hessians:** Second-order optimization methods need $\nabla^2 L$
- **Jacobians:** Understanding how outputs change with respect to inputs
- **JVPs/VJPs:** Efficient directional derivatives without materializing full Jacobians

### Key API

```python
torch.autograd.grad(
    outputs,       # what to differentiate
    inputs,        # with respect to what
    grad_outputs,  # weighting (required if outputs is not scalar)
    create_graph,  # True if you need higher-order derivatives
    retain_graph,  # True if you call grad multiple times
) -> tuple of gradients
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: First and Second Derivatives of Known Functions (12 points)

**[Coding]** Verify autograd against known derivatives.

For $f(x) = \sin(x^2)$:
- $f'(x) = 2x\cos(x^2)$
- $f''(x) = 2\cos(x^2) - 4x^2\sin(x^2)$

1. (4 points) Compute $f'(x)$ using `autograd.grad` at $x = 1.0$. Compare with the analytical value.
2. (4 points) Compute $f''(x)$ using two chained `autograd.grad` calls. Compare with analytical.
3. (4 points) Compute $f'''(x)$ using three chained calls. Derive the analytical formula first, then verify.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Compute f', f'', f''' for f(x) = sin(x^2) at x = 1.0

""" END OF THIS PART """

## Part 2: Jacobian Computation (15 points)

**[Coding]** Compute the full Jacobian matrix of a vector-valued function.

For $f: \mathbb{R}^n \to \mathbb{R}^m$, the Jacobian is:

$$J_{ij} = \frac{\partial f_i}{\partial x_j}$$

1. (5 points) Implement `compute_jacobian(f, x)` that computes the full Jacobian matrix by calling `autograd.grad` once per output dimension (using one-hot `grad_outputs`).

2. (5 points) Test with $f(x) = Ax + b$ where $A$ is a known $3 \times 4$ matrix. The Jacobian should be exactly $A$.

3. (5 points) Test with a nonlinear function: $f(x) = [x_1^2 + x_2, \sin(x_1 x_2)]$ at $x = [1, 2]$. Compute analytically and compare.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_jacobian(f, x):
    """
    Compute the Jacobian matrix of f at x.
    
    Args:
        f: callable, maps (n,) -> (m,)
        x: tensor of shape (n,) with requires_grad=True
    
    Returns:
        J: tensor of shape (m, n)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: Hessian Computation (15 points)

**[Coding]** Compute the Hessian matrix of a scalar-valued function.

For $f: \mathbb{R}^n \to \mathbb{R}$, the Hessian is:

$$H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$$

1. (8 points) Implement `compute_hessian(f, x)` by:
   - First computing $\nabla f$ using `autograd.grad` with `create_graph=True`
   - Then computing each row of the Hessian by differentiating each component of $\nabla f$

2. (4 points) Test with $f(x) = x^T A x$ where $A$ is symmetric. The Hessian should be $2A$.

3. (3 points) Verify the Hessian is symmetric for a nonlinear function $f(x_1, x_2) = x_1^2 x_2 + \sin(x_1 x_2)$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_hessian(f, x):
    """
    Compute the Hessian matrix of scalar function f at x.
    
    Args:
        f: callable, maps (n,) -> scalar
        x: tensor of shape (n,) with requires_grad=True
    
    Returns:
        H: tensor of shape (n, n)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Jacobian-Vector Product (JVP) (12 points)

**[Coding]** A JVP computes $Jv$ without materializing the full Jacobian $J$.

For $f: \mathbb{R}^n \to \mathbb{R}^m$ and a vector $v \in \mathbb{R}^n$:

$$\text{JVP} = Jv = \lim_{\epsilon \to 0} \frac{f(x + \epsilon v) - f(x)}{\epsilon}$$

In PyTorch, this is computed using **forward-mode AD**. However, we can also compute it using `autograd.grad` with a trick.

1. (6 points) Implement JVP using `autograd.grad`:
   - Compute $y = f(x)$
   - Compute $s = \sum_i v_i \cdot x_i$ (a scalar depending on $x$ and $v$)
   - The gradient $\nabla_x s = v$
   - Use `autograd.grad(y, x, grad_outputs=...)` strategically
   
   **Hint:** Actually, the simplest approach is `autograd.grad(y, x, grad_outputs=v_out)` gives VJP ($v^T J$). For JVP, use the dual number trick: compute $f(x + \epsilon v)$ and differentiate w.r.t. $\epsilon$ at $\epsilon = 0$.

2. (6 points) Verify by comparing JVP with $J \cdot v$ (using your Jacobian function from Part 2).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_jvp(f, x, v):
    """
    Compute the Jacobian-vector product J @ v.
    
    Args:
        f: callable, maps (n,) -> (m,)
        x: tensor of shape (n,)
        v: tensor of shape (n,)
    
    Returns:
        jvp: tensor of shape (m,)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 5: Vector-Jacobian Product (VJP) (10 points)

**[Coding]** A VJP computes $v^T J$ without materializing $J$. This is what `autograd.grad` naturally computes.

1. (5 points) Implement VJP using a single call to `autograd.grad` with appropriate `grad_outputs`.

2. (5 points) Verify by comparing with $v^T J$ using your Jacobian function.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_vjp(f, x, v):
    """
    Compute the vector-Jacobian product v^T @ J.
    
    Args:
        f: callable, maps (n,) -> (m,)
        x: tensor of shape (n,)
        v: tensor of shape (m,)  -- note: m, not n
    
    Returns:
        vjp: tensor of shape (n,)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 6: Hessian-Vector Product (12 points)

**[Coding]** A Hessian-vector product $Hv$ can be computed efficiently without materializing $H$.

**Method:** For scalar $f(x)$:
1. Compute $g = \nabla f(x)$ with `create_graph=True`
2. Compute $\nabla_x (g \cdot v) = Hv$ with another `autograd.grad` call

This is $O(n)$ instead of $O(n^2)$.

1. (6 points) Implement `hessian_vector_product(f, x, v)`.

2. (6 points) Verify against $H \cdot v$ using your full Hessian from Part 3.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def hessian_vector_product(f, x, v):
    """
    Compute the Hessian-vector product H @ v efficiently.
    
    Args:
        f: callable, maps (n,) -> scalar
        x: tensor of shape (n,)
        v: tensor of shape (n,)
    
    Returns:
        hvp: tensor of shape (n,)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 7: Application — Newton's Method (14 points)

**[Coding]** Use your Hessian and gradient computations to implement Newton's method for optimization.

**Newton's update:** $x_{k+1} = x_k - H^{-1} \nabla f(x_k)$

1. (4 points) Implement Newton's method using your `compute_hessian` function from Part 3. Use `torch.linalg.solve` instead of explicit inverse.

2. (5 points) Minimize the Rosenbrock function: $f(x_1, x_2) = (1 - x_1)^2 + 100(x_2 - x_1^2)^2$. Start from $x_0 = (-1, 1)$. The minimum is at $(1, 1)$.

3. (5 points) Compare the number of iterations to reach $\|\nabla f\| < 10^{-6}$ between Newton's method and gradient descent with lr=0.001.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def newtons_method(f, x0, max_iter=100, tol=1e-6):
    """
    Minimize f using Newton's method.
    
    Returns:
        x: final point
        trajectory: list of (x, f(x)) pairs
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 8: Application — Neural Network Loss Landscape (10 points)

**[Coding]** Compute the Hessian of a small neural network's loss.

1. (5 points) Create a tiny network: Linear(2, 3) → ReLU → Linear(3, 1). Flatten all parameters into a single vector. Compute the Hessian of the MSE loss at a single data point $(x, y)$ where $x = [1, 2]$, $y = [1]$.

2. (5 points) Compute the eigenvalues of the Hessian. Are they all non-negative? What does this tell you about the loss landscape at this point? (Note: the loss landscape of neural networks is generally non-convex, so negative eigenvalues are expected.)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Neural network loss Hessian

""" END OF THIS PART """